# Truck Special Generators Review and Recommendations for Travel Model 1.7

This notebook reviews truck special generator production estimates from CSF2TDM and the ALACC Bi-County Model. The analysis:

1. Reviews ALACC special generator inputs and identifies their geographic coverage.
2. Compares truck production estimates between ALACC and CSF2TDM.
3. Summarizes truck demand associated with California Gateways in CSF2TDM.
4. Recommends special generator production inputs for Travel Model 1.7.

The primary outcome is a recommended set of truck special generator production estimates and production-rate assumptions for forecasting.


In [ ]:
import pandas as pd 
import numpy as np 

import geopandas as gpd
import openmatrix as omx

import warnings
from pandas.errors import PerformanceWarning

warnings.filterwarnings(
    "ignore",
    category=PerformanceWarning,
)

In [ ]:
#inputs: 
alacc_sg = "../data/external/ccta/PORT_SG_2015.csv" #Special generators demand AlaCC Model
sw_production = "../data/interim/matrix_projection/production_attraction_summaries/truck_trip_generation_zone.csv"
sw_special_generators = "../data/interim/matrix_projection/production_attraction_summaries/truck_trip_generation_special_generators.csv"
landuse = "../data/external/mtc/2023_TM161_IPA_35/landuse/tazData.csv"

#output: 
special_generators_rates = "../data/interim/notebook_outputs/tm17_special_generators_production_rates.csv"

In [ ]:
def add_total_row(df: pd.DataFrame, label: str = "Total") -> pd.DataFrame:
    """Return a copy of df with a totals row appended."""

    result = df.copy()

    total_row = {}

    for col in result.columns:
        if pd.api.types.is_numeric_dtype(result[col]):
            total_row[col] = result[col].sum(skipna=True)
        else:
            total_row[col] = ""

    total_df = pd.DataFrame([total_row], index=[label])

    return pd.concat([result, total_df])

# Bi-county Special Generation Data

In [ ]:
# Explore PORT_SG_2015 from the ALACC Bi-County Model.
# Original source: PORT_SG_2015.DBF (exported to CSV using Cube).
# Goal: inspect the special generator truck production inputs and
# summarize TAZs with non-zero productions.

PORT_SG_FIELDS = ["taz", "SMALL", "MEDIUM", "COMBO"]

# Read and clean input file
port_sg = (
    pd.read_csv(alacc_sg, header=None)
    .dropna()
)

port_sg.columns = PORT_SG_FIELDS

# Convert TAZ to integer and use as index
port_sg["taz"] = port_sg["taz"].astype(float).astype(int)
port_sg = port_sg.set_index("taz")

# Total truck productions across all vehicle classes, and then across all TAZs
port_sg["total"] = port_sg[["SMALL", "MEDIUM", "COMBO"]].sum(axis=1)
bcm_port_sg = add_total_row(port_sg.query("total > 0"))

print("Table 1. Special Generator Truck Productions from ALACC")
bcm_port_sg.style.format("{:,.0f}")

Note: A visual review of these TAZ locations indicates that all are located within the Port of Oakland.

# State Wide Model 

### Comparison of Port of Oakland Truck Productions: ALACC vs. CSF2TDM

The ALACC Bi-County Model represents Port of Oakland truck activity through special generator inputs. In CSF2TDM, comparable truck demand is represented through two distinct sources:

1. **TAZ-level truck productions** generated by the statewide truck trip generation framework.
2. **California Gateway demand**, which represents truck activity associated with major freight gateways and logistics facilities.

To create a comparable estimate of Port of Oakland truck productions, both components are included in the CSF2TDM estimate. TM1.6 TAZs 965, 966, and 988 were identified through manual map review as approximately representing the same geographic area covered by the ALACC Port of Oakland special generator.

The table below reports truck productions from the TAZ-based model and California Gateway model separately, as well as their combined totals. The comparison is intended as a reasonableness check between the ALACC and CSF2TDM estimates and should not be interpreted as an exact geographic correspondence.

In [ ]:
# TM1.6 TAZs judged to approximately represent the Port of Oakland area
port_of_oakland_tazs = [965, 966, 988]

# ---------------------------------------------------------------------
# TAZ-level truck productions (CSF2TDM)
# ---------------------------------------------------------------------

sw_generation = (
    pd.read_csv(sw_production)
    .set_index("TAZ1454")
)

# Aggregate productions by truck class
sw_generation["small"] = (
    sw_generation.filter(regex=r"^LT.*production$").sum(axis=1)
)
sw_generation["medium"] = (
    sw_generation.filter(regex=r"^MT.*production$").sum(axis=1)
)
sw_generation["large"] = (
    sw_generation.filter(regex=r"^HT.*production$").sum(axis=1)
)

# TAZ productions within the Port of Oakland area
production_port_of_oakland = sw_generation.loc[
    sw_generation.index.isin(port_of_oakland_tazs),
    ["small", "medium", "large"]
]

# ---------------------------------------------------------------------
# California International Gateways (special generators)
# ---------------------------------------------------------------------

gateways = pd.read_csv(sw_special_generators)

# Aggregate TLN productions by truck class
gateways["small"] = gateways.filter(regex=r"^LT.*production$").sum(axis=1)
gateways["medium"] = gateways.filter(regex=r"^MT.*production$").sum(axis=1)
gateways["large"] = gateways.filter(regex=r"^HT.*production$").sum(axis=1)

# ---------------------------------------------------------------------
# Combine TAZ and TLN demand
# ---------------------------------------------------------------------
# In CSF2TDM, Port of Oakland truck demand is represented by both
# TAZ-based production estimates and TLN-based gateway demand.

sw_port_of_oakland = (
    production_port_of_oakland
    .merge(
        gateways[["TAZ1454", "small", "medium", "large"]],
        left_index=True,
        right_on="TAZ1454",
        how="left",
        suffixes=("_taz", "_gateways"),
    )
    .set_index("TAZ1454")
    .fillna(0)
)

# Total demand across all components
sw_port_of_oakland["total"] = sw_port_of_oakland.sum(axis=1)
sw_port_of_oakland.index = sw_port_of_oakland.index.astype(int)

# Add regional total row for comparison with ALACC special generator totals
sw_port_of_oakland = add_total_row(sw_port_of_oakland)

print("Table 2. CSF2TDM Port of Oakland Demand (TAZ + TLN)")
sw_port_of_oakland.style.format("{:,.0f}")

### California Gateway Truck Demand in CSF2TDM

This section summarizes truck demand associated with California Gateways in CSF2TDM. These locations represent major freight gateways and logistics facilities that generate truck demand separately from the statewide TAZ-based truck trip generation framework.

For each gateway, the table reports:

- Truck productions generated by the TAZ-based model.
- Truck productions generated by the California Gateway model.
- Combined totals across both sources.

Only TAZs containing a California Gateway are included. The purpose of this analysis is to identify the relative contribution of TAZ-based and gateway-based demand at major freight locations and to document the magnitude of truck activity represented by each gateway in CSF2TDM.

In [ ]:
# Join TAZ-level truck productions with California Gateway productions
# for all TAZs containing a gateway.
all_gateways = (
    sw_generation
    .merge(
        gateways,
        on="TAZ1454",
        how="inner",
        suffixes=("_taz", "_tln")
    )
    .set_index(["TAZ1454", "zone_name"])
)

# Keep truck production summaries by source
all_gateways = all_gateways[
    [
        "small_taz",
        "medium_taz",
        "large_taz",
        "small_tln",
        "medium_tln",
        "large_tln",
    ]
]

# Combined truck demand from TAZ and Gateway components
all_gateways["total"] = all_gateways.sum(axis=1)

# Add regional total row for summary reporting
all_gateways = add_total_row(all_gateways)

print("Table 3. CSF2TDM Truck Demand at California Gateway Locations")
all_gateways.style.format("{:,.0f}")

# TM-1.7 Updates Recommendation for Special Generators. 
We recommend using the ALACC Port of Oakland production estimates because the model was developed in collaboration with the Port of Oakland and reflects conditions in Alameda County. Furthermore, the resulting trip estimates are generally consistent in magnitude with truck activity documented in the appendix of the Port of Oakland Truck Queuing Study report, providing additional support for their use (reference: https://www.oaklandseaport.com/wp-content/uploads/2025/06/OHTBW-Final-EIR_App-I_Truck-Queuing-Study.pdf).

For all other special generators, we recommend using the CSF2TDM 2020 production estimates, as no alternative data sources or validation references were identified.

For forecasting, production rates are calculated by dividing estimated truck productions by total employment (TOTEMP) within each special generator TAZ. Consistent with the assumption used in the ALACC model, truck productions are assumed to equal truck attractions. The same assumption is applied to all special generators

In [ ]:
# Read TAZ data to extract tota employment 'TOTEMP'
taz_land_use = pd.read_csv(landuse)
region_employment = taz_land_use["TOTEMP"].sum()

In [ ]:
# Summary Production Table: Special Generator, TAZ, Medium/Large Charge production, source, TOTAL employment, Production Rate
df = all_gateways[["medium_tln", "large_tln"]].round(0).reset_index()

# Remove Total row
df = df[df["index"] != "Total"].copy()

# Extract tuple components
df["taz1454"] = df["index"].str[0]
df["special_generation"] = df["index"].str[1]

df = (
    df.assign(source="CSF2TDM")
    .rename(
        columns={
            "medium_tln": "medium_production",
            "large_tln": "large_production",
        }
    )
    [
        [
            "special_generation",
            "taz1454",
            "medium_production",
            "large_production",
            "source",
        ]
    ]
)
# Replace Port of Oakland data with ALACC estimate
mask = df["special_generation"] == "PORT OF OAKLAND"
df.loc[mask, "medium_production"] = bcm_port_sg.loc["Total", "MEDIUM"]
df.loc[mask, "large_production"] = bcm_port_sg.loc["Total", "COMBO"]
df.loc[mask, "source"] = "ALACC Bicounty model"

# Merge employment data 
df["region_employment"] = region_employment
df["medium_production_rate"] = (df["medium_production"]/df["region_employment"]).round(8)
df["large_production_rate"] = (df["large_production"]/df["region_employment"]).round(8)
df.to_csv(special_generators_rates)